# Celeste Classic Reinforcement Learning — Results
**CS 4824 · Machine Learning · Spring 2026**  
**Team:** Akhil Madipalli (akhimadi), Maryam Malik (maryam23), Jorge Manuel Torre (jmt1006)

## Project description

We trained agents to play room 0 of *Celeste Classic* (the original PICO-8 version) using reinforcement learning, and ran a controlled comparison of five method variants against a random baseline:

1. **Random** — uniform action sampling. Floor.
2. **Behavioral Cloning (BC)** — supervised imitation of a Tool-Assisted Speedrun (TAS) replay through the room. Action-weighted cross-entropy with kinematic-feature noise augmentation.
3. **Plain DQN** — trial-and-error RL with a 15-step milestone reward ramp. Network is a plain MLP `Input(87) → 256 → 256 → 128 → 15`.
4. **Dueling DQN + curiosity bonus** — same DQN backbone but with the Wang et al. value/advantage decomposition and a Bellemare-style counts-based intrinsic motivation reward (+0.5 per newly-visited 4-pixel cell, +0.2 for cells visited fewer than ten times globally).
5. **Hybrid (BC + DQN)** — DQN warm-started with TAS expert transitions in a protected 20% partition of the replay buffer.
6. **Curriculum** — six-stage backwards spawn schedule, advancing when ≥50% of the last 100 episodes complete.

All methods use the same environment (an 87-dim semantic-tile state, see *Setup* below), the same 15 discrete actions, and the same hyperparameters where applicable.

## The headline finding

**State representation matters more than algorithm choice.** Replacing raw PICO-8 tile IDs (0–255) with a five-class semantic encoding (air / solid / spike / out-of-bounds / other) raised plain DQN's completion rate from about 8% to roughly 60% with no other change. None of the algorithmic enhancements we tried — Dueling, curiosity, BC, hybrid, or curriculum — beat plain DQN with the perception fix.

## Notebook structure (QAC)

Each results section follows a **Question → Analysis → Conclusion** format:
- **Question:** what are we testing or measuring?
- **Analysis:** training curves, evaluation metrics, agent behavior.
- **Conclusion:** what we learned, and how it informs the next approach.

Every cell loads from artifacts already in the repo (`docs/comparison/`, `docs/*.png`, `docs/*.gif`). No retraining is required to reproduce any plot below.

## Setup

Imports and a quick load of the comparison summary.

In [ ]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

# Notebook lives in notebooks/, so paths go up one level.
REPO = Path('..').resolve()
DOCS = REPO / 'docs'
COMPARISON = DOCS / 'comparison'
RUNS = REPO / 'runs'

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Headline numbers from compare_all.py (ε=0.05, 100 episodes per method).
with open(COMPARISON / 'summary.json') as f:
    summary = json.load(f)

rows = []
for run_id, v in summary.items():
    rows.append({
        'method': run_id,
        'arch': v['arch'],
        'completion %': round(v['completion_rate'] * 100, 1),
        'died': v['deaths'],
        'timeout': v['timeouts'],
        'mean_reward': round(v['mean_reward'], 1),
        'mean_height': round(v['mean_height'], 1),
    })
df = pd.DataFrame(rows).sort_values('completion %', ascending=False).reset_index(drop=True)
df

## The problem

Celeste Classic is a precision platformer built in PICO-8. Room 0 looks simple — climb to the top of one screen — but requires a frame-perfect wall-jump-dash sequence at the upper ledge to clear the exit threshold (`y < -4`). The agent has 500 environment steps per episode and must learn:

- **Long-horizon credit assignment:** 50–100+ correct decisions before any terminal reward.
- **Frame-perfect inputs:** the final dash-jump sequence is ~5–10 frames where each input matters.
- **Sparse reward:** a +500 terminal bonus that ε-greedy exploration almost never observes by chance.

These are well-known failure modes for standard DQN, so we expected the algorithmic comparison to be the interesting axis. The perception-fix finding came from going down a level — looking at how the env represented the world to the agent in the first place.

## Section 1 — The perception fix

**Question:** Does state representation matter more than algorithm choice in this domain?

**Analysis.** We ran the same DQN training loop with two different tile encodings:

- *Raw tile IDs* (`v3_r8`): each tile in the 9×9 view is its raw PICO-8 ID (0–255). A spike is ID 17, decoration grass is ID 18 — numerically adjacent.
- *Semantic classes* (`dqn_r1`): each tile is mapped to one of five categories: air (0.0), solid (1.0), spike (-1.0), other (0.5), out-of-bounds sentinel (-2.0).

Below we plot the rolling completion rate from each training run.

In [ ]:
def load_history(run_id):
    """Load training pickle. Tries flat layout (runs/{run_id}_training.pkl) first,
    then folder layout (runs/{run_id}/{run_id}_training.pkl), then docs fallback."""
    candidates = [
        RUNS / f'{run_id}_training.pkl',
        RUNS / run_id / f'{run_id}_training.pkl',
        DOCS / f'training_{run_id}.pkl',
    ]
    for p in candidates:
        if p.exists():
            with open(p, 'rb') as f:
                return pickle.load(f)
    raise FileNotFoundError(f'No training pickle for {run_id}')

def rolling_completion_rate(completions_log, window=50):
    arr = np.array([1 if c else 0 for c in completions_log], dtype=float)
    if len(arr) < window:
        return np.array([])
    return np.convolve(arr, np.ones(window) / window, mode='valid') * 100

fig, ax = plt.subplots(figsize=(10, 4))
any_loaded = False
for run_id, color, label in [
    ('v3_r8', 'tab:red', 'v3_r8 — raw tile IDs'),
    ('dqn_r1', 'tab:green', 'dqn_r1 — semantic encoding'),
]:
    try:
        hist = load_history(run_id)
        rate = rolling_completion_rate(hist['completions_log'])
        ax.plot(range(50, 50 + len(rate)), rate, color=color, label=label, linewidth=2)
        any_loaded = True
    except (FileNotFoundError, KeyError):
        print(f'{run_id} pickle not found — see saved curve image instead')
ax.set_xlabel('Episode')
ax.set_ylabel('Rolling 50-ep completion rate (%)')
ax.set_title('Same DQN, same hyperparameters — only the tile encoding changed')
if any_loaded:
    ax.legend()
plt.tight_layout()
plt.show()

**Conclusion.** Same DQN, same hyperparameters, same reward function, same training budget — only the tile encoding changed.

- `v3_r8` (raw tile IDs) plateaus around **8% peak** completion rate and then collapses.
- `dqn_r1` (semantic encoding) hits its first completions at episode 1500 and **sustains 30–60%** completion rate through the rest of training.

The takeaway: with raw tile IDs the agent had no semantic signal — a spike (ID 17) and a decoration (ID 18) were numerically as similar as air (ID 0) and a wall (ID 1). The network had no way to learn what was deadly versus landable. The semantic encoding fix is fifteen lines of `if/elif` in `_get_obs()`. It's the single most impactful change in the project.

Every subsequent section uses the semantic encoding.

## Section 2 — Plain DQN (the winner)

**Question:** What does plain DQN with semantic state achieve under controlled evaluation?

**Analysis.** `dqn_r1` is plain DQN — no Dueling architecture, no curiosity bonus, no expert data, no curriculum. Just trial-and-error Q-learning with the milestone reward ramp and the semantic tile encoding. 5000 episodes.

In [ ]:
display(Image(filename=str(DOCS / 'dqn_r1_curve.png')))

In [ ]:
if 'dqn_r1' in summary:
    v = summary['dqn_r1']
    print(f"dqn_r1 — eval @ ε=0.05, {v['episodes']} episodes")
    print(f"  Completion: {v['completes']}/{v['episodes']} ({v['completion_rate']*100:.1f}%)")
    print(f"  Died:       {v['deaths']}/{v['episodes']}")
    print(f"  Timeout:    {v['timeouts']}/{v['episodes']}")
    print(f"  Mean reward:  {v['mean_reward']:.1f} ± {v['std_reward']:.1f}")
    print(f"  Mean height:  {v['mean_height']:.1f}  (lower = closer to exit; exit at y < -4)")

**Conclusion.** `dqn_r1` reliably learned to wall-jump up the left side of the room, traverse the upper section, and execute a final wall-jump-dash through the exit threshold.

Headline numbers:
- **57–68% completion** under controlled evaluation (single seed; per-run vs batched evals fall within ±10 points).
- **86% peak** completion rate during training (episode 1950 window).
- **35.7% lifetime** training rate (1783 / 5000 episodes completed).
- No catastrophic collapse — sustained 22+/50 completion rate from episode 2000 through 5000.

The bimodal eval reward distribution (~2700–3350 for completions, ~20–300 for failures) reflects that when this policy works it works cleanly, and when it fails it usually fails early to ε-noise.

## Section 3 — Behavioral Cloning

**Question:** Can a policy learn to play this room from imitation alone?

**Analysis.** `bc_r2` is supervised classification on 66 (state, action) pairs from a TAS any% replay through room 0. Action-weighted cross-entropy loss with Gaussian noise on kinematic features (σ=0.02) to discourage overfit to the exact trajectory.

In [ ]:
display(Image(filename=str(DOCS / 'bc_r2_curve.png')))

In [ ]:
if 'bc_r2' in summary:
    v = summary['bc_r2']
    print(f"bc_r2 — eval @ ε=0.05, {v['episodes']} episodes")
    print(f"  Completion: {v['completes']}/{v['episodes']} ({v['completion_rate']*100:.1f}%)")
    print(f"  Died:       {v['deaths']}/{v['episodes']}")
    print(f"  Timeout:    {v['timeouts']}/{v['episodes']}  (deterministic loop)")
    print(f"  Mean height:  {v['mean_height']:.1f}")

**Conclusion.** 0% completion. The dominant failure mode is **timeouts, not deaths** — the BC agent reaches a state slightly off the TAS trajectory, has no recovery signal in the supervised loss, picks whatever action looked closest to right, drifts further, and locks into a deterministic loop at mid-room.

This is the textbook BC failure mode: **compounding error / covariate shift**. Pure imitation provides demonstrations of optimal behavior but no signal for recovery from off-distribution states. The 0% number isn't a failed experiment — it's the empirical anchor for *why* hybrid imitation+RL methods exist as a class.

## Section 4 — Dueling DQN + curiosity bonus

**Question:** Do the standard DQN improvements (Dueling architecture, intrinsic motivation reward) help in this domain?

**Analysis.** `v3_r9` is the same training pipeline as `dqn_r1` plus two additions:

1. **Dueling DQN** (Wang et al. 2016): network splits into a value stream `V(s)` and an advantage stream `A(s, a)`, combined as `Q = V + (A − mean(A))`. Tends to help when many states have action-redundant outcomes.
2. **Counts-based intrinsic motivation** (Bellemare et al. 2016): +0.5 reward each time the agent visits a new (x // 4, y // 4) cell within an episode, +0.2 bonus if the cell has been visited fewer than ten times globally.

In [ ]:
display(Image(filename=str(DOCS / 'v3_r9.png')))

In [ ]:
if 'v3_r9' in summary:
    v = summary['v3_r9']
    print(f"v3_r9 — eval @ ε=0.05, {v['episodes']} episodes")
    print(f"  Completion: {v['completes']}/{v['episodes']} ({v['completion_rate']*100:.1f}%)")
    print(f"  Died:       {v['deaths']}/{v['episodes']}  (often at upper-ledge zone)")
    print(f"  Timeout:    {v['timeouts']}/{v['episodes']}")

**Conclusion.** Adding Dueling architecture + curiosity bonus actually *hurt* — `v3_r9` peaked at 64% in training, then collapsed mid-run. Eval gave 37–49%, well below `dqn_r1`'s 57–68%.

Possible reasons:
- The curiosity bonus rewards visiting *new* (x, y) cells. Once the agent finds the optimal path, those rewards mislead — pulling it toward unexplored corners away from the trajectory it should be exploiting. Likely caused the mid-training collapse.
- Dueling's pitch (decomposing V from A when actions are redundant) doesn't apply to Celeste room 0, where every frame demands a precise input. Dense milestone rewards make V(s) easily learnable through Q directly.

Note that `dqn_r1` vs `v3_r9` differs on two axes simultaneously (architecture and reward shaping), so the 20-point gap can't be cleanly attributed to either factor alone — see the future-work section.

## Section 5 — Hybrid (BC + DQN)

**Question:** Does a DQN warm-started with TAS expert transitions outperform plain DQN?

**Analysis.** `hybrid_r2` pre-loads the 66 TAS transitions into a protected partition of the replay buffer (20% expert / 80% online, expert never evicted) and trains DQN normally on top.

In [ ]:
display(Image(filename=str(DOCS / 'hybrid_r2_curve.png')))

In [ ]:
if 'hybrid_r2' in summary:
    v = summary['hybrid_r2']
    print(f"hybrid_r2 — eval @ ε=0.05, {v['episodes']} episodes")
    print(f"  Completion: {v['completes']}/{v['episodes']} ({v['completion_rate']*100:.1f}%)")
    print(f"  Died:       {v['deaths']}/{v['episodes']}  (100% deaths — striking failure mode)")
    print(f"  Timeout:    {v['timeouts']}/{v['episodes']}")

**Implementation issue we identified post-hoc.** Looking at `train_hybrid.py:104`:

```python
transitions.append((obs, action_idx, 0.0, next_obs, done))
```

Every TAS reward is hardcoded to `0.0`. Q-learning bootstraps `Q(s, a) ≈ r + γ · max Q(s', a')`. With `r = 0` for 20% of every batch, the network is being *pulled toward zero* on the (state, action) pairs along the optimal trajectory — exactly opposite to what the env's milestone rewards would teach.

**Conclusion.** 0% completion under our specific implementation, with 100% deaths (the agent walks straight into spikes — worse than random, which had 85% deaths). This is a real implementation bug, not a property of imitation+RL methods in general. A correct implementation would replay the TAS through the env to capture the actual reward stream rather than zeroing them out. We did not have time to fix and re-run before the deadline; see *Future work*.

## Section 6 — Curriculum learning

**Question:** Does a backwards spawn curriculum unlock the policy for the full level?

**Analysis.** `curriculum_r5` runs six stages with progressively lower spawn-y values, each advancing when ≥50% of the last 100 episodes complete:

| Stage | Spawn y | Outcome |
|---|---|---|
| 1 | 15 | advanced |
| 2 | 30 | advanced |
| 3 | 50 | advanced |
| 4 | 60 | timeout |
| 5 | 71 | timeout |
| 6 | 96 (full level) | timeout |

In [ ]:
display(Image(filename=str(DOCS / 'curriculum_r5_curve.png')))

In [ ]:
if 'curriculum_r5' in summary:
    v = summary['curriculum_r5']
    print(f"curriculum_r5 — eval @ ε=0.05, {v['episodes']} episodes")
    print(f"  Completion: {v['completes']}/{v['episodes']} ({v['completion_rate']*100:.1f}%)")
    print(f"  Died:       {v['deaths']}/{v['episodes']}")
    print(f"  Timeout:    {v['timeouts']}/{v['episodes']}")

**Conclusion.** 0% completion at full-level eval, despite the agent successfully learning to complete from y=15, y=30, and y=50 spawns during stages 1–3.

The failure mode is a classic **curriculum-overfit**: the policy specializes to the curriculum's intermediate spawn distributions rather than to the underlying environment. When the agent is later spawned at y=96 (the actual room start), it has never seen those early-room states during training and fails to navigate them.

Compare with `dqn_r1`, which trained from y=96 every episode for 5000 episodes and reached 57–68%. Straight RL solved what the curriculum could not — given enough time on the full task, the agent learns the full task.

## Section 7 — Final comparison

**Question:** Across all five methods plus the random baseline, which won and by how much?

In [ ]:
df_show = pd.read_csv(COMPARISON / 'comparison_table.csv')
df_show

In [ ]:
display(Image(filename=str(COMPARISON / 'completion_bar.png')))

In [ ]:
display(Image(filename=str(COMPARISON / 'outcome_breakdown.png')))

In [ ]:
display(Image(filename=str(COMPARISON / 'height_distribution.png')))

In [ ]:
# Side-by-side gameplay — one episode per method, run in lockstep, sorted by completion.
display(Image(filename=str(DOCS / 'comparison.gif')))

**Conclusion.** Five method variants. Only one beat the random baseline meaningfully. `dqn_r1` (plain DQN with semantic state) won by 20+ percentage points over the next-best variant.

The death/timeout split per method is signal, not noise:

- `dqn_r1` mixed: some early ε-noise deaths, some "almost made it" timeouts at the upper ledge
- `v3_r9` dies more, often at the upper-ledge zone (curiosity pulled it off-path)
- `random` walks into spikes
- `bc_r2` freezes in a deterministic loop and runs out of time (88% timeouts)
- `curriculum_r5` freezes harder (97% timeouts) — never learned the full-level start
- `hybrid_r2` actively walks into spikes (100% deaths) — anti-trained by zero-reward expert transitions
- `v3_r8` freezes like BC despite being a DQN — raw tile IDs gave the network nothing useful to learn from

**The dominant factor across all comparisons was state representation, not algorithm choice.** Every "enhancement" we tried (BC, curriculum, hybrid, Dueling, curiosity) underperformed plain DQN with the same state encoding.

## Discussion and caveats

Honest limits of what we ran:

1. **Hybrid had a real implementation bug** (zero-reward expert transitions). The 0% number reflects our specific implementation, not the general capability of imitation+RL methods. A correct implementation might recover meaningfully.
2. **`dqn_r1` vs `v3_r9` isn't a clean single-variable ablation.** They differ on architecture (plain vs Dueling DQN) and reward shaping (no curiosity vs curiosity bonus). Decomposing the 20-point gap would have required another run.
3. **Single-seed comparisons.** Each method ran once with one random seed. Variance estimates would require multiple seeds — we observed roughly ±10 percentage points of variance between per-run and batched evaluations of the same checkpoint.
4. **Single-room evaluation.** All comparisons are on Celeste room 0. Generalization to other rooms or full-game completion is future work. Prior art ([effdotsh/Celeste-Bot](https://github.com/effdotsh/Celeste-Bot), a genetic algorithm in a Unity recreation) does solve the full game, suggesting population-based or on-policy methods with curiosity (PPO + RND) are probably the right algorithm class for the harder rooms.

## Future work

Each direction is a single runnable experiment on the existing infrastructure.

- **Fix hybrid and re-run.** Replay the TAS through the env to capture real rewards rather than zeroing them. Whether a corrected hybrid matches plain DQN, beats it, or still underperforms is the most interesting open question we have.
- **Multiple seeds.** Three seeds per method would give us confidence bands on every result and let us test whether the 20-point gap is significant at p < 0.05. ~30 GPU-hours.
- **Generalization to other rooms.** Rooms 5+ require dash chains, rooms 12+ have wind, rooms 20+ have falling blocks. Whether the perception-fix dominance holds across them is open.
- **Single-variable architectural ablation.** Run plain DQN + curiosity bonus to cleanly decompose the dqn_r1 vs v3_r9 gap into architectural and reward-shaping components.
- **Larger expert dataset for BC.** Adding 5–10 hand-recorded human playthroughs to the existing 66 TAS transitions would test whether BC's failure is fundamental (covariate shift) or just data-limited.

## References

- **Pyleste** — Python PICO-8 emulator: [github.com/CelesteClassic/Pyleste](https://github.com/CelesteClassic/Pyleste)
- **Celeste Classic** — original game: [celesteclassic.github.io](https://celesteclassic.github.io/)
- **TAS database** — expert demonstration source: [celesteclassic.github.io/tasdatabase/classic](https://celesteclassic.github.io/tasdatabase/classic/)
- **DQN** — Mnih et al., *Playing Atari with Deep Reinforcement Learning*, [arXiv:1312.5602](https://arxiv.org/abs/1312.5602)
- **Dueling DQN** — Wang et al., *Dueling Network Architectures for Deep Reinforcement Learning*, [arXiv:1511.06581](https://arxiv.org/abs/1511.06581)
- **Counts-based curiosity** — Bellemare et al., *Unifying Count-Based Exploration and Intrinsic Motivation*, [arXiv:1606.01868](https://arxiv.org/abs/1606.01868)
- **Celeste-Bot (GA)** — independent agent that solves the full game: [github.com/effdotsh/Celeste-Bot](https://github.com/effdotsh/Celeste-Bot)
- **Project devlog** — full debugging story: [DEVLOG.md](https://github.com/jmtorr3/celeste-rl/blob/master/DEVLOG.md)